# ВКР: полный запуск в Google Colab

Этот notebook ещё не выполнен. Он обучит все модели, включая **Keras/TensorFlow**, и скачает итоговый ZIP. GPU не нужен. Выберите Runtime → Run all. В первой ячейке загрузите ZIP проекта. Обучение запускается отдельным Python-процессом, поэтому установленные версии используются независимо от состояния notebook-ядра.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, json
print('Выберите исходный ZIP проекта')
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Нужно выбрать один ZIP проекта')
archive = Path(archives[0])
base = Path.cwd() / 'vkr_work'
base.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (base / name).resolve()
        if not target.is_relative_to(base.resolve()):
            raise ValueError('Недопустимый путь в архиве')
    z.extractall(base)
PROJECT = base / 'vkr_composites'
assert (PROJECT / 'train.py').exists(), 'В архиве нет vkr_composites/train.py'
print('Проект:', PROJECT)


## Установка библиотек

Установка выполняется только в облачной среде Colab. На компьютер ничего устанавливать не нужно.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
print('Библиотеки установлены')


## Полное обучение и проверки

Сначала EDA и регрессии, затем Keras. Результаты не подменяются числами старого проекта. Если процесс завершится ошибкой, будет показан конец журнала и скачивание окончательного ZIP не начнётся.

In [ ]:
log_path = PROJECT / 'training_log.txt'
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-u', 'train.py'], cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
print(log_path.read_text()[-10000:])
if result.returncode:
    raise RuntimeError('Обучение завершилось ошибкой; скопируйте последние строки журнала')
manifest = json.loads((PROJECT / 'reports/run_manifest.json').read_text())
assert manifest['neural_network']['status'] == 'trained', 'ИНС не обучена'
subprocess.run([sys.executable, 'verify.py'], cwd=PROJECT, check=True)
subprocess.run([sys.executable, 'make_research_notebook.py'], cwd=PROJECT, check=True)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## Скачивание готового проекта

Архив содержит обученную Keras-сеть, скейлеры, регрессии, графики, метрики, выполненный обзорный notebook и приложение. Распакуйте его на компьютере в новую папку; запускайте START_WINDOWS.cmd из неё. README описывает также исходный этап подготовки: фактический статус последнего расчёта — в run_manifest.json.

In [ ]:
verification = json.loads((PROJECT / 'reports/app_verification.json').read_text())
assert verification['status'] == 'passed'
output = Path.cwd() / 'vkr_completed.zip'
with zipfile.ZipFile(output, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(PROJECT.rglob('*')):
        if p.is_file() and not any(part in ('__pycache__', '.venv', '.git') for part in p.parts):
            z.write(p, Path('vkr_composites') / p.relative_to(PROJECT))
with zipfile.ZipFile(output) as z:
    assert z.testzip() is None
print('Готово:', output.name, 'Сеть обучена, приложение проверено.')
files.download(str(output))
